# Loan Status Prediction: EDA

This notebook documents the main exploratory checks for the loan status dataset: target balance, missing values, numeric distributions, categorical target rates, and a first leakage sanity check.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
DATA_PATH = PROJECT_ROOT / 'Data' / 'loan_data.csv'
df = pd.read_csv(DATA_PATH)
df.shape

## Dataset Overview

The dataset contains 45,000 rows and 14 columns. The target column is `loan_status`.

In [ ]:
display(df.head())
display(df.info())
display(df.isna().sum().sort_values(ascending=False))

## Target Balance

The target is imbalanced: about 22.2% positive class and 77.8% negative class. This is why the modeling pipeline uses stratified splitting, class balancing, and business-cost-aware threshold tuning.

In [ ]:
target_share = df['loan_status'].value_counts(normalize=True).sort_index()
display(target_share)
sns.barplot(x=target_share.index.astype(str), y=target_share.values)
plt.title('Loan status target balance')
plt.xlabel('loan_status')
plt.ylabel('share')
plt.show()

## Numeric Features

Numeric features should be inspected for heavy tails and unrealistic outliers before final model interpretation. Income, loan amount, interest rate, and loan-percent-income are especially important for credit-risk behavior.

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.drop('loan_status')
display(df[numeric_cols].describe().T)
df[numeric_cols].hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()

## Categorical Features And Target Rates

`previous_loan_defaults_on_file` is the most suspicious field: in the current dataset, category `Yes` has a 0.0 positive target rate. This can be a valid business rule, but it must be confirmed as information available before the loan decision.

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    summary = df.groupby(col)['loan_status'].agg(rows='count', target_rate='mean').sort_values('target_rate', ascending=False)
    display(summary)

plt.figure(figsize=(8, 4))
sns.barplot(data=df, x='previous_loan_defaults_on_file', y='loan_status', estimator='mean')
plt.title('Target rate by previous loan default flag')
plt.ylabel('target rate')
plt.show()

## EDA Conclusions

- The target is moderately imbalanced, so accuracy alone is not enough.
- The dataset has no missing values in the current file, but preprocessing still includes imputers for robust inference.
- `previous_loan_defaults_on_file` dominates the target split and should be treated as a leakage/business-rule risk until confirmed.
- Next modeling reports should compare performance with and without suspicious features.